# Colab Experiment: CLIP-space Concept Poisoning

Goal: apply CLIP-space concept poisoning toward a target text concept, measure semantic shift, then run a classification proxy train-after-poison benchmark.

## 1. Setup Colab / GitHub repo

This cell clones the repo when running from a fresh Colab runtime and installs dependencies.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

# If this notebook is opened directly in Colab, clone the GitHub repo first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional Google Drive output

GitHub stores source code only. Use Drive if you want results to persist after the Colab runtime resets.

In [ ]:
# Optional: copy final results to Google Drive after the run.
# Set SAVE_TO_DRIVE = True if you want persistent result storage.
SAVE_TO_DRIVE = False
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/adversarial-data-protection-results"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    print("Drive results dir:", DRIVE_RESULTS_DIR)

## 3. Run experiment

In [ ]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from scripts.run_experiment import setup_dirs, collect_clean_tensors, train_classifier, build_protected_tensors
from src.datasets import get_cifar10
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_victim_resnet18
from src.techniques.nightshade import _normalize_clip, get_text_embedding, load_clip_model
from src.visualization import plot_before_after

# Smoke defaults. Concept poisoning uses CLIP ViT-B/32 and is a Nightshade-style proxy.
SUBSET_SIZE = 500
BATCH_SIZE = 64
EPSILON = 0.05
TARGET_CONCEPT = "a photo of a cat"
BASELINE_EPOCHS = 2
VICTIM_EPOCHS = 2
PGD_STEPS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
setup_dirs()

train_loader, test_loader = get_cifar10(subset_size=SUBSET_SIZE, batch_size=BATCH_SIZE)
clean_x, clean_y = collect_clean_tensors(train_loader)

print("Training clean baseline victim...")
baseline = train_classifier(lambda: get_victim_resnet18(device=device), train_loader, BASELINE_EPOCHS, device)
baseline_acc = evaluate(baseline, test_loader, device)
print("baseline_clean_test_accuracy:", baseline_acc)

print("Loading CLIP and generating concept-poisoned data...")
clip_model, _ = load_clip_model("ViT-B/32", device)
protected_x = build_protected_tensors(
    "concept_poisoning",
    clean_x,
    clean_y,
    device,
    epsilon=EPSILON,
    clip_model=clip_model,
    pgd_steps=PGD_STEPS,
)

@torch.no_grad()
def compute_clip_target_shift(x_orig, x_protected, target_text, batch_size=64):
    target = get_text_embedding(clip_model, target_text, device)
    before_values, after_values = [], []
    clip_model.eval()
    for start in range(0, x_orig.size(0), batch_size):
        xo = F.interpolate(x_orig[start:start+batch_size].to(device), size=(224, 224), mode="bilinear", align_corners=False)
        xp = F.interpolate(x_protected[start:start+batch_size].to(device), size=(224, 224), mode="bilinear", align_corners=False)
        fo = F.normalize(clip_model.encode_image(_normalize_clip(xo)).float(), dim=1)
        fp = F.normalize(clip_model.encode_image(_normalize_clip(xp)).float(), dim=1)
        before_values.append(F.cosine_similarity(fo, target.expand_as(fo)).cpu())
        after_values.append(F.cosine_similarity(fp, target.expand_as(fp)).cpu())
    before = torch.cat(before_values)
    after = torch.cat(after_values)
    return round(before.mean().item(), 4), round(after.mean().item(), 4), round((after - before).mean().item(), 4)

clip_before, clip_after, clip_delta = compute_clip_target_shift(clean_x, protected_x, TARGET_CONCEPT)

protected_loader = DataLoader(TensorDataset(protected_x, clean_y), batch_size=BATCH_SIZE, shuffle=True)
print("Training victim on concept-poisoned train set...")
victim = train_classifier(lambda: get_victim_resnet18(device=device), protected_loader, VICTIM_EPOCHS, device)
clean_acc, asr = compute_attack_success_rate(victim, test_loader, device)

metrics = {
    "technique": "concept_poisoning",
    "victim_model": "ResNet-18",
    "subset_size": SUBSET_SIZE,
    "epsilon": EPSILON,
    "target_concept": TARGET_CONCEPT,
    "baseline_clean_test_accuracy": baseline_acc,
    "protected_clean_test_accuracy": clean_acc,
    "accuracy_drop": round(baseline_acc - clean_acc, 4),
    "asr_proxy": asr,
    "clip_target_similarity_before": clip_before,
    "clip_target_similarity_after": clip_after,
    "clip_target_similarity_delta": clip_delta,
    "psnr": compute_psnr(clean_x, protected_x),
    "ssim": compute_ssim(clean_x, protected_x),
    "linf": compute_linf(clean_x, protected_x),
}
print(metrics)

os.makedirs("results/tables", exist_ok=True)
pd.DataFrame([metrics]).to_csv("results/tables/concept_poisoning_experiment.csv", index=False)

sample_idx = 0
plot_before_after(clean_x[sample_idx], protected_x[sample_idx], "concept_poisoning")
to_pil = torchvision.transforms.ToPILImage()
to_pil(clean_x[sample_idx]).save("results/protected_samples/concept_poisoning_original.png")
to_pil(protected_x[sample_idx]).save("results/protected_samples/concept_poisoning_protected.png")
print("Saved results/tables/concept_poisoning_experiment.csv and sample images.")

## 4. Optional copy results to Drive

In [ ]:
# Optional copy to Drive.
if SAVE_TO_DRIVE:
    import shutil
    target = Path(DRIVE_RESULTS_DIR)
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("results", target / "results", dirs_exist_ok=True)
    print("Copied results to", target / "results")